# **Project Name**    -



##### **Project Type**    - EDA/Regression/Classification/Unsupervised
##### **Contribution**    - Individual
##### **Team Member 1 - Anant tripathi
##### **Team Member 2 -**
##### **Team Member 3 -**
##### **Team Member 4 -**

# **Project Summary -**

Write the summary here within 500-600 words.

# **GitHub Link -**

Provide your GitHub Link here.

# **Problem Statement**


**Write Problem Statement Here.**

# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 15 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





6. You may add more ml algorithms for model creation. Make sure for each and every algorithm, the following format should be answered.


*   Explain the ML Model used and it's performance using Evaluation metric Score Chart.


*   Cross- Validation & Hyperparameter Tuning

*   Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

*   Explain each evaluation metric's indication towards business and the business impact pf the ML model used.




















# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries
# Built-in tools for file paths and text processing
from pathlib import Path
import re

# Data handling and calculations
import numpy as np
import pandas as pd

# Charts
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical tests
from scipy import stats

# Data preprocessing and model pipelines
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    GridSearchCV,
    GroupShuffleSplit,
    RandomizedSearchCV,
    StratifiedGroupKFold,
    cross_val_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

# Model evaluation
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

# Saving and loading the fitted model
import joblib

# Apply a consistent style to charts in later cells
sns.set_theme(style="whitegrid")

print("Libraries imported successfully.")
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

### Dataset Loading

In [ ]:
# Load Dataset
# Search common locations for the supplied CSV file.
data_candidates = [
    Path("dataset-2.csv"),
    Path("../dataset-2.csv"),
    Path("/content/dataset-2.csv"),
]
data_path = next((path for path in data_candidates if path.exists()), None)

# Stop with a clear message if the file is not in any expected location.
if data_path is None:
    raise FileNotFoundError(
        "dataset-2.csv was not found. Place it beside the notebook "
        "or update data_candidates with its location."
    )

# Load the original data and keep a copy for later cleaning.
try:
    df_raw = pd.read_csv(data_path, low_memory=False)
except (OSError, UnicodeDecodeError, pd.errors.ParserError) as exc:
    raise RuntimeError(f"Could not read {data_path}: {exc}") from exc

# Trim accidental spaces from column names.
df_raw.columns = df_raw.columns.astype(str).str.strip()

# Use df for initial inspection; preserve df_raw unchanged.
df = df_raw.copy()

print(f"Loaded file: {data_path}")
print("Dataset loaded successfully.")

### Dataset First View

In [ ]:
# Dataset First Look
# Preview the data while omitting direct identifiers and customer keys.
identifier_columns = {"ID", "Customer_ID", "Name", "SSN"}
preview_columns = [column for column in df.columns if column not in identifier_columns]
display(df.loc[:, preview_columns].head())
display(df.loc[:, preview_columns].sample(n=min(5, len(df)), random_state=42))


### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
# Read the dataset dimensions: rows first, then columns.
row_count, column_count = df.shape

print(f"Number of rows: {row_count:,}")
print(f"Number of columns: {column_count:,}")

# Check how many distinct customers and target classes are present.
if "Customer_ID" in df.columns:
    print(f"Unique customers: {df['Customer_ID'].nunique():,}")

if "Credit_Score" in df.columns:
    print("\nCredit score counts:")
    display(df["Credit_Score"].value_counts(dropna=False).to_frame("Count"))

### Dataset Information

In [ ]:
# Dataset Info
# Show each column's data type and non-missing-value count.
df.info(show_counts=True)

# Summarize numeric and categorical columns.
display(df.describe(include="all").T)

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
# Count rows that are identical across every column.
full_duplicate_count = int(df.duplicated().sum())
print(f"Exact duplicate rows: {full_duplicate_count:,}")

# Check whether a customer-month pair occurs more than once.
# Repeated Customer_ID values alone are expected because customers have monthly records.
if {"Customer_ID", "Month"}.issubset(df.columns):
    duplicate_customer_months = int(
        df.duplicated(subset=["Customer_ID", "Month"]).sum()
    )
    print(f"Duplicate Customer_ID + Month pairs: {duplicate_customer_months:,}")

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
# Count missing values and calculate their percentage for every column.
missing_summary = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percent": (df.isna().mean() * 100).round(2),
})

# Show columns with the most missing values first.
missing_summary = missing_summary.sort_values(
    by="Missing_Count", ascending=False
)

display(missing_summary)

In [ ]:
# Visualizing the missing values
# Keep only columns that actually contain missing values.
missing_to_plot = missing_summary.loc[
    missing_summary["Missing_Count"] > 0, "Missing_Count"
]

plt.figure(figsize=(10, 4))

if missing_to_plot.empty:
    # Avoid an empty chart when the dataset has no blank cells.
    plt.text(
        0.5, 0.5,
        "No blank or null values found in the clean dataset.",
        ha="center", va="center", fontsize=12,
    )
    plt.axis("off")
else:
    # Plot only columns with missing values so the chart stays readable.
    sns.barplot(
        x=missing_to_plot.values,
        y=missing_to_plot.index,
        color="steelblue",
    )
    plt.title("Missing Values by Column")
    plt.xlabel("Missing-value count")
    plt.ylabel("Column")

plt.tight_layout()
plt.show()

### What did you know about your dataset?

Answer Here 

The dataset contains 100,000 monthly customer records and 28 variables. Each of 12,500 customers has eight monthly records, so records from the same customer are related. The target variable is Credit_Score, with three classes: Standard (53,174 records), Poor (28,998), and Good (17,828). The target is moderately imbalanced, so model evaluation should include per-class recall and macro F1, not accuracy alone.
The file has no blank cells or exact duplicate rows. However, it includes explicit category values such as NM and No Data; these should not be confused with empty values. Customer identifiers repeat by design across months, so they should be used to keep each customer’s records together when splitting data, not as predictive features. Names and SSNs should also be excluded from model features.
The dataset includes financial, account, credit-history, and payment-behavior fields.

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
# Display the dataset's column names and dimensions.
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

for number, column in enumerate(df.columns, start=1):
    print(f"{number:>2}. {column}")

In [ ]:
# Dataset Describe
# Summarize numeric and categorical columns.
display(df.describe(include="all").T)

### Variables Description

Answer Here
 The dataset contains customer identifiers, monthly records, personal details, financial information, credit history, and payment behavior. Variables such as `Annual_Income`, `Monthly_Inhand_Salary`, and `Outstanding_Debt` describe financial status. Account and credit variables include `Num_Bank_Accounts`, `Num_Credit_Card`, `Num_of_Loan`, `Credit_Mix`, and `Credit_Utilization_Ratio`. Payment behavior is represented by fields such as `Delay_from_due_date`, `Num_of_Delayed_Payment`, and `Payment_of_Min_Amount`. `Credit_Score` is the target variable, with the classes Poor, Standard, and Good. `ID`, `Customer_ID`, `Name`, and `SSN` are identifiers or personal information, so they should not be used as predictive features. The dataset does not include a data dictionary, so units and precise definitions should be confirmed.

### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
# Build a readable summary without printing personal or identifier values.
identifier_columns = {"ID", "Customer_ID", "Name", "SSN"}

def summarize_values(column):
    values = df[column].dropna()
    unique_count = df[column].nunique(dropna=False)

    # Hide names and identifier values; only report their unique counts.
    if column in identifier_columns:
        examples = "Values hidden (identifier or personal information)"

    # Summarize numeric columns with a range instead of listing many values.
    elif pd.api.types.is_numeric_dtype(df[column]):
        if values.empty:
            examples = "No non-missing values"
        else:
            examples = (
                f"Min={values.min():,.2f}; "
                f"Median={values.median():,.2f}; "
                f"Max={values.max():,.2f}"
            )

    # List every value for small categories, with counts.
    elif unique_count <= 10:
        counts = df[column].value_counts(dropna=False)
        examples = "; ".join(
            f"{value}: {count:,}" for value, count in counts.items()
        )

    # For high-cardinality text, show only its five most common values.
    else:
        top_values = df[column].value_counts(dropna=False).head(5)
        examples = "; ".join(
            f"{value}: {count:,}" for value, count in top_values.items()
        )

    return pd.Series({
        "Data Type": str(df[column].dtype),
        "Unique Values": unique_count,
        "Missing Values": int(df[column].isna().sum()),
        "Examples / Summary": examples,
    })

# Create one table covering every column.
unique_values_table = pd.DataFrame({
    column: summarize_values(column) for column in df.columns
}).T

display(unique_values_table)

In [ ]:
# Plot category counts for meaningful categorical variables.
# Identifiers and high-cardinality loan descriptions are intentionally excluded.
categorical_columns_to_plot = [
    "Credit_Score",
    "Credit_Mix",
    "Payment_of_Min_Amount",
    "Payment_Behaviour",
    "Occupation",
]
categorical_columns_to_plot = [
    column for column in categorical_columns_to_plot if column in df.columns
]

# Create one horizontal count plot per categorical column.
fig, axes = plt.subplots(
    nrows=len(categorical_columns_to_plot),
    ncols=1,
    figsize=(10, 4 * len(categorical_columns_to_plot)),
)

# Make axes iterable even if only one chart is created.
if len(categorical_columns_to_plot) == 1:
    axes = [axes]

for column, ax in zip(categorical_columns_to_plot, axes):
    category_order = df[column].value_counts().index

    sns.countplot(
        data=df,
        y=column,
        order=category_order,
        ax=ax,
        color="steelblue",
    )
    ax.set_title(f"Category Counts: {column}")
    ax.set_xlabel("Number of records")
    ax.set_ylabel(column)

plt.tight_layout()
plt.show()

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Write your code to make your dataset analysis ready.
# Work on a copy so the original loaded data remains available for comparison.
df = df_raw.copy()

# Remove accidental spaces around column names.
df.columns = df.columns.astype(str).str.strip()

# Convert these numeric-looking columns to numeric data types.
numeric_columns = [
    "ID", "Customer_ID", "Month", "Age", "SSN", "Annual_Income",
    "Monthly_Inhand_Salary", "Num_Bank_Accounts", "Num_Credit_Card",
    "Interest_Rate", "Num_of_Loan", "Delay_from_due_date",
    "Num_of_Delayed_Payment", "Changed_Credit_Limit",
    "Num_Credit_Inquiries", "Outstanding_Debt",
    "Credit_Utilization_Ratio", "Credit_History_Age",
    "Total_EMI_per_month", "Amount_invested_monthly", "Monthly_Balance",
]

# Clean text columns and convert common blank markers into missing values.
text_columns = df.select_dtypes(include=["object", "string"]).columns
for column in text_columns:
    df[column] = df[column].astype("string").str.strip()
    df[column] = df[column].replace(
        r"(?i)^\s*(?:NA|N/A|NULL|NONE|___)?\s*$",
        pd.NA,
        regex=True,
    )

# Convert numeric text safely. Any unexpected unparseable value becomes missing.
for column in numeric_columns:
    if column in df.columns:
        cleaned_text = (
            df[column]
            .astype("string")
            .str.replace(",", "", regex=False)
            .str.replace("_", "", regex=False)
            .str.strip()
        )
        df[column] = pd.to_numeric(cleaned_text, errors="coerce")

# Normalize target spelling and keep "NM" as a meaningful category.
df["Credit_Score"] = df["Credit_Score"].astype("string").str.title()
df["Payment_of_Min_Amount"] = df["Payment_of_Min_Amount"].replace(
    {"NM": "Not Mentioned"}
)

# Check that the target contains the expected classes.
expected_scores = {"Poor", "Standard", "Good"}
actual_scores = set(df["Credit_Score"].dropna().unique())

if not actual_scores.issubset(expected_scores):
    raise ValueError(f"Unexpected Credit_Score values found: {actual_scores - expected_scores}")

# Report what changed and verify the final data shape and missing values.
print("Data wrangling complete.")
print("Shape after cleaning:", df.shape)
print("Exact duplicate rows:", int(df.duplicated().sum()))
print("Missing cells after type conversion:", int(df.isna().sum().sum()))
print("Credit score counts:")
display(df["Credit_Score"].value_counts(dropna=False).to_frame("Count"))

### What all manipulations have you done and insights you found?

Answer Here.
I preserved the original dataset in df_raw and performed the cleaning steps on a copy called df. I trimmed spaces from column names and text values, converted numeric-looking columns to numeric data types, and converted common blank markers into missing values. I renamed NM in Payment_of_Min_Amount to Not Mentioned, keeping it distinct from Yes and No. I also checked the target values, missing cells, and exact duplicate rows.
No rows were removed during this cleaning step. The dataset has no blank cells or exact duplicate rows. Each of the 12,500 customers appears in eight monthly records; I kept these records because they are part of the dataset’s structure. I will use Customer_ID to keep a customer’s records together when splitting the data, while excluding it, ID, Name, and SSN from model features.
The target is Credit_Score, with three categories: Standard (53,174 records), Poor (28,998), and Good (17,828). Standard is the most frequent class, so model evaluation should include macro F1 and per-class recall in addition to accuracy. Values such as No Data remain explicit categories rather than being mistaken for blank cells.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
# Chart - 1 visualization code
# Chart - 1 visualization code
score_order = ["Poor", "Standard", "Good"]

# Count each target class and preserve a consistent display order.
score_counts = (
    df["Credit_Score"]
    .value_counts()
    .reindex(score_order, fill_value=0)
)

# Draw one bar per credit score class.
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(score_counts.index, score_counts.values, color=["#d95f5f", "#e6b94f", "#58a878"])

# Label each bar with its count and percentage of the dataset.
for bar, count in zip(bars, score_counts.values):
    percentage = count / len(df) * 100
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{count:,}\n({percentage:.1f}%)",
        ha="center",
        va="bottom",
    )

ax.set_title("Distribution of Credit Score Classes")
ax.set_xlabel("Credit score")
ax.set_ylabel("Number of records")
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

Answer Here.
A bar chart makes it easy to compare the number of records in each target class.

##### 2. What is/are the insight(s) found from the chart?

Answer Here
Standard is the most common class with 53,174 records (53.2%). Poor has 28,998 (29.0%), and Good has 17,828 (17.8%). The target classes are moderately imbalanced.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here
This helps identify the need to evaluate performance for each class, using measures such as macro F1 and per-class recall alongside accuracy. If the model mostly predicts Standard, it could miss Poor or Good customers, leading to poor risk assessment or unsuitable guidance.

#### Chart - 2

In [ ]:
# Chart - 2 visualization code
# Chart - 2 visualization code
age_values = df["Age"].dropna()

# Plot the age distribution and mark its median.
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(age_values, bins=25, kde=True, ax=ax, color="steelblue")

median_age = age_values.median()
ax.axvline(
    median_age,
    color="darkred",
    linestyle="--",
    label=f"Median age: {median_age:.0f}",
)

ax.set_title("Customer Age Distribution")
ax.set_xlabel("Age")
ax.set_ylabel("Number of records")
ax.legend()
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

Answer Here.
A histogram shows the range and concentration of ages in the dataset. The median line gives a simple reference for the typical record.

##### 2. What is/are the insight(s) found from the chart?

Answer Here.In the supplied data, ages range from 14 to 56, with a median of 33. The dataset therefore includes a broad range of ages rather than representing only one narrow age group

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here
The age profile helps describe who is represented in the data and can guide checks for gaps in coverage. Age should not be used as a simplistic measure of creditworthiness; inappropriate reliance on it could lead to unfair treatment.

#### Chart - 3

In [ ]:
# Chart - 3 visualization code
# Chart - 3 visualization code

# Use a log scale to make the wide income range easier to inspect.
income_log10 = np.log10(df["Annual_Income"].dropna().clip(lower=1))

fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(income_log10, bins=30, kde=True, ax=ax, color="teal")

ax.set_title("Annual Income Distribution (log10 scale)")
ax.set_xlabel("log10 of annual income")
ax.set_ylabel("Number of records")
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

Answer Here.
A histogram shows the shape and spread of a numeric variable. The log scale makes the income distribution easier to read when values span a wide range.

##### 2. What is/are the insight(s) found from the chart?

Answer Here
Annual income ranges from about 7,006 to 179,987 dataset units. Its mean is about 50,505, while its median is about 36,997, indicating a right-skewed distribution: a smaller number of high-income records pull the mean upward.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here
Understanding the income distribution helps describe customers’ financial profiles and informs model preprocessing. Income should be considered alongside debt and payment history; using it alone could lead to unfair or inaccurate assessments. The dataset does not specify the currency or units, so confirm these before interpreting the amounts as real-world values.

#### Chart - 4

In [ ]:
# Chart - 4 visualization code
# Chart - 4 visualization code

# Count the records in each credit-mix category in a fixed order.
credit_mix_order = ["Bad", "Standard", "Good"]
credit_mix_counts = (
    df["Credit_Mix"]
    .value_counts()
    .reindex(credit_mix_order, fill_value=0)
)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(
    credit_mix_counts.index,
    credit_mix_counts.values,
    color=["#d95f5f", "#e6b94f", "#58a878"],
)

# Label each bar with its count and dataset percentage.
for bar, count in zip(bars, credit_mix_counts.values):
    percentage = count / len(df) * 100
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{count:,}\n({percentage:.1f}%)",
        ha="center",
        va="bottom",
    )

ax.set_title("Credit Mix Category Distribution")
ax.set_xlabel("Credit mix")
ax.set_ylabel("Number of records")
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

Answer Here.
A bar chart is a clear way to compare the number of records in each credit-mix category.

##### 2. What is/are the insight(s) found from the chart?

Answer Here 
Standard is the most frequent category, with 45,848 records (45.8%). Good has 30,384 (30.4%), and Bad has 23,768 (23.8%).

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here
The chart shows that all three categories have substantial representation for analysis. Credit mix may help describe customer profiles, but the counts alone do not show how it relates to the target. Before using it for prediction, verify that it is available at prediction time and is not derived from Credit_Score; otherwise, it could leak target information into the model.

#### Chart - 5

In [ ]:
# Chart - 5 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 6

In [ ]:
# Chart - 6 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 7

In [ ]:
# Chart - 7 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 8

In [ ]:
# Chart - 8 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 9

In [ ]:
# Chart - 9 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 10

In [ ]:
# Chart - 10 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 11

In [ ]:
# Chart - 11 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 12

In [ ]:
# Chart - 12 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 13

In [ ]:
# Chart - 13 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 14 - Correlation Heatmap

In [ ]:
# Correlation Heatmap visualization code
# Correlation Heatmap visualization code

# Select numeric predictors and exclude IDs and the categorical target.
correlation_columns = [
    "Annual_Income",
    "Monthly_Inhand_Salary",
    "Monthly_Balance",
    "Num_of_Loan",
    "Outstanding_Debt",
    "Interest_Rate",
    "Num_Credit_Inquiries",
    "Credit_History_Age",
    "Delay_from_due_date",
    "Num_of_Delayed_Payment",
    "Num_Bank_Accounts",
]

# Calculate pairwise Pearson correlations between the selected numeric variables.
correlation_matrix = df[correlation_columns].corr(numeric_only=True)

# Hide the upper triangle so each variable pair appears only once.
upper_triangle_mask = np.triu(
    np.ones_like(correlation_matrix, dtype=bool),
    k=1,
)

# Draw the annotated heatmap.
fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(
    correlation_matrix,
    mask=upper_triangle_mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=0.5,
    ax=ax,
)

ax.set_title("Correlation Heatmap of Selected Numeric Features")
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

Answer Here.
A correlation heatmap summarizes the direction and strength of relationships between numeric variables. I excluded identifiers and the categorical target because treating Poor, Standard, and Good as numeric would impose an arbitrary distance between classes.

##### 2. What is/are the insight(s) found from the chart?

Answer Here
Annual income and monthly in-hand salary are very strongly correlated (about 0.998), so they carry nearly the same linear information in this dataset. Other notable relationships include number of loans and outstanding debt (about 0.639), and interest rate and credit inquiries (about 0.633). These are associations; the heatmap does not show that one variable causes another.

#### Chart - 15 - Pair Plot

In [ ]:
# Pair Plot visualization code
# Pair Plot visualization code

# Choose a small set of numeric features to keep the plot readable.
pairplot_columns = [
    "Age",
    "Annual_Income",
    "Outstanding_Debt",
    "Delay_from_due_date",
    "Credit_History_Age",
]

# Drop incomplete rows for these plotted columns and take a reproducible sample.
pairplot_data = df[pairplot_columns + ["Credit_Score"]].dropna()
pairplot_data = pairplot_data.sample(
    n=min(1500, len(pairplot_data)),
    random_state=42,
)

# Show each feature's distribution and the pairwise scatter plots, colored by score.
pair_grid = sns.pairplot(
    data=pairplot_data,
    vars=pairplot_columns,
    hue="Credit_Score",
    hue_order=score_order,
    corner=True,
    diag_kind="hist",
    plot_kws={"alpha": 0.35, "s": 14},
)

pair_grid.fig.suptitle(
    "Pair Plot of Numeric Features by Credit Score",
    y=1.02,
)
plt.show()

##### 1. Why did you pick the specific chart?

Answer Here.A pair plot combines feature distributions with scatter plots for pairs of features. Coloring points by credit score helps inspect whether combinations of numeric variables show class patterns. The sample keeps plotting responsive with 100,000 records.


##### 2. What is/are the insight(s) found from the chart?

Answer Here
The pair plot should show overlap between score classes rather than a clean separation by a single feature pair. In the dataset, Poor rows tend to have higher payment delays, while Good rows tend to have longer credit histories. Income and debt also show differing score-group medians, but the classes are not perfectly separated.

## ***5. Hypothesis Testing***

### Based on your chart experiments, define three hypothetical statements from the dataset. In the next three questions, perform hypothesis testing to obtain final conclusion about the statements through your code and statistical testing.

Answer Here.I test three relationships suggested by the charts: annual income between Good and Poor score groups, association between Credit Mix and Credit Score, and payment delay across score groups. I use a significance level of 0.05 and a Bonferroni-adjusted threshold of 0.05 / 3 = 0.0167 because three tests are being performed.

In [ ]:
# Return the most frequent nonmissing value for each customer.
def safe_mode(series):
    modes = series.mode(dropna=True)
    return modes.iloc[0] if not modes.empty else pd.NA

# Summarize the eight monthly records into one row per customer.
customer_summary = (
    df.groupby("Customer_ID", as_index=False)
      .agg(
          Credit_Score=("Credit_Score", safe_mode),
          Annual_Income=("Annual_Income", "median"),
          Credit_Mix=("Credit_Mix", safe_mode),
          Delay_from_due_date=("Delay_from_due_date", "mean"),
      )
      .dropna()
)

print(f"Customers included in the tests: {len(customer_summary):,}")

# Format extremely small p-values clearly instead of displaying numerical zero.
def format_p_value(p_value):
    if p_value == 0:
        return "< 1e-300 (numerical underflow)"
    return f"{p_value:.3g}"

### Hypothetical Statement - 1

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Answer Here.
Customers with a modal Good score do not have higher annual income than customers with a modal Poor score.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
# Compare customer-level income distributions for Good and Poor score groups.
good_income = customer_summary.loc[
    customer_summary["Credit_Score"] == "Good", "Annual_Income"
]
poor_income = customer_summary.loc[
    customer_summary["Credit_Score"] == "Poor", "Annual_Income"
]

# Use a one-sided Mann-Whitney U test to check whether Good incomes tend to be higher.
income_test = stats.mannwhitneyu(
    good_income,
    poor_income,
    alternative="greater",
    method="asymptotic",
)

# Calculate rank-biserial correlation as an effect-size estimate.
income_effect = (
    2 * income_test.statistic / (len(good_income) * len(poor_income)) - 1
)

print(f"Good group median income: {good_income.median():,.2f}")
print(f"Poor group median income: {poor_income.median():,.2f}")
print(f"Mann-Whitney U statistic: {income_test.statistic:,.0f}")
print(f"p-value: {format_p_value(income_test.pvalue)}")
print(f"Rank-biserial effect size: {income_effect:.3f}")

##### Which statistical test have you done to obtain P-Value?

Answer Here.
A one-sided Mann–Whitney U test comparing customer-level annual income for the Good and Poor score groups.

##### Why did you choose the specific statistical test?

Answer Here. Annual income can be skewed, so this test compares the two independent groups without assuming normally distributed data. I summarized each customer’s repeated monthly records first to avoid treating all eight rows per customer as independent.

### Hypothetical Statement - 2

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Answer Here.
Credit Mix and Credit Score are independent.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
# Create a customer-level contingency table for Credit Mix and Credit Score.
mix_order = ["Bad", "Standard", "Good"]
score_order = ["Poor", "Standard", "Good"]

mix_score_table = pd.crosstab(
    customer_summary["Credit_Mix"],
    customer_summary["Credit_Score"],
).reindex(index=mix_order, columns=score_order, fill_value=0)

# Test whether the two categorical variables are independent.
chi2_stat, p_value, degrees_of_freedom, expected_counts = (
    stats.chi2_contingency(mix_score_table)
)

# Calculate Cramér's V to describe the strength of the association.
sample_size = mix_score_table.to_numpy().sum()
cramers_v = np.sqrt(
    chi2_stat
    / (sample_size * min(
        mix_score_table.shape[0] - 1,
        mix_score_table.shape[1] - 1,
    ))
)

display(mix_score_table)
print(f"Chi-square statistic: {chi2_stat:,.2f}")
print(f"Degrees of freedom: {degrees_of_freedom}")
print(f"p-value: {format_p_value(p_value)}")
print(f"Cramér's V: {cramers_v:.3f}")

##### Which statistical test have you done to obtain P-Value?

Answer Here.
A chi-square test of independence.

##### Why did you choose the specific statistical test?

Answer Here.
Credit Mix and Credit Score are both categorical variables. The chi-square test checks whether their observed category counts differ from the counts expected if the variables were independent. The customer-level table treats each customer as one observation.

### Hypothetical Statement - 3

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Answer Here.
Customer-level mean delay distributions are the same for the Poor, Standard, and Good score groups.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
# Collect each customer's mean delay for each modal score group.
score_order = ["Poor", "Standard", "Good"]

delay_groups = [
    customer_summary.loc[
        customer_summary["Credit_Score"] == score,
        "Delay_from_due_date",
    ].to_numpy()
    for score in score_order
]

# Compare the three groups without assuming normally distributed delay values.
delay_test = stats.kruskal(*delay_groups)

# Estimate the size of the group difference.
total_customers = sum(len(group) for group in delay_groups)
epsilon_squared = delay_test.statistic / (total_customers - 1)

print("Mean customer delay by score:",
      [round(group.mean(), 2) for group in delay_groups])
print(f"Kruskal-Wallis H statistic: {delay_test.statistic:,.2f}")
print(f"p-value: {format_p_value(delay_test.pvalue)}")
print(f"Epsilon-squared effect size: {epsilon_squared:.3f}")

##### Which statistical test have you done to obtain P-Value?

Answer Here.
The Kruskal–Wallis test.

##### Why did you choose the specific statistical test?

Answer Here.
It compares a numeric variable across three independent groups without requiring normally distributed values. The test can show that at least one group differs, but it does not identify which pairs differ.

## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
# Handling Missing Values & Missing Value Imputation
# Count missing values after data cleaning.
missing_counts = df.isna().sum().sort_values(ascending=False)
missing_columns = missing_counts[missing_counts > 0]

if missing_columns.empty:
    print("No blank or null values found after cleaning.")
else:
    print("Columns with missing values:")
    display(missing_columns.to_frame("Missing_Count"))

# Define imputers for use inside the model preprocessing pipeline.
# They should be fit on training data only.
numeric_imputer = SimpleImputer(strategy="median")
categorical_imputer = SimpleImputer(strategy="most_frequent")

print("Numeric fallback: median imputation")
print("Categorical fallback: most-frequent-category imputation")

#### What all missing value imputation techniques have you used and why did you use those techniques?

Answer Here.
The supplied CSV has no blank or null values after cleaning, so I did not fill values in the analysis dataframe. I retained NM as “Not Mentioned” and No Data as an explicit category. For robustness, the model pipeline uses median imputation for numeric features and most-frequent-category imputation for categorical features. The imputers are fit within the training pipeline to avoid learning from validation or test data.

### 2. Handling Outliers

In [ ]:
# Handling Outliers & Outlier treatments
# Check selected numeric features for observations outside the 1.5 × IQR range.
outlier_columns = [
    "Annual_Income",
    "Outstanding_Debt",
    "Total_EMI_per_month",
    "Monthly_Balance",
    "Delay_from_due_date",
]

outlier_results = []

for column in outlier_columns:
    values = df[column].dropna()

    # Use the interquartile range to flag unusually distant values.
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outlier_count = int(((values < lower_bound) | (values > upper_bound)).sum())

    outlier_results.append({
        "Feature": column,
        "Lower_IQR_Bound": lower_bound,
        "Upper_IQR_Bound": upper_bound,
        "Flagged_Count": outlier_count,
        "Flagged_Percent": round(outlier_count / len(values) * 100, 2),
    })

display(pd.DataFrame(outlier_results))

##### What all outlier treatment techniques have you used and why did you use those techniques?

Answer Here.
I used the 1.5 × IQR rule to flag potential outliers for review. I did not automatically remove or cap them: high income, debt, or payment values may be valid customer records, and deleting them could remove meaningful risk information. The model pipeline uses RobustScaler for numeric features, which is less sensitive to extreme values than standard scaling. The flagged values should be checked against the data definitions before any treatment.

### 3. Categorical Encoding

In [ ]:
# Encode your categorical columns
# Create a OneHotEncoder for categorical predictors.
# Handle unknown categories so future records do not cause an encoding error.
try:
    categorical_encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
    )
except TypeError:
    # Compatibility with older scikit-learn versions.
    categorical_encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse=False,
    )

print("One-hot encoder is ready for the model preprocessing pipeline.")

#### What all categorical encoding techniques have you used & why did you use those techniques?

Answer Here.I use one-hot encoding for categorical predictors such as Credit_Mix, Payment_of_Min_Amount, and Payment_Behaviour. This represents categories as separate indicator features without implying a numeric ranking. handle_unknown="ignore" lets the model accept a category that did not appear during training. The encoder should be fit inside the model pipeline, using training data only.

### 4. Textual Data Preprocessing
(It's mandatory for textual dataset i.e., NLP, Sentiment Analysis, Text Clustering etc.)

#### 1. Expand Contraction

In [ ]:
# Expand Contraction


# This dataset has no free-form text field for NLP processing.
NLP_TEXT_COLUMNS = []

def note_nlp_step(step, reason):
    """Document why an NLP step is skipped for this structured dataset."""
    print(f"{step}: not applicable — {reason}")

note_nlp_step("Expand contractions", "there are no free-form sentences.")

#### 2. Lower Casing

In [ ]:
# Lower Casing
note_nlp_step(
    "Lower casing",
    "loan product matching is handled case-insensitively during feature creation."
)

#### 3. Removing Punctuations

In [ ]:
# Remove Punctuations
note_nlp_step(
    "Removing punctuation",
    "punctuation such as the hyphen in 'Credit-Builder Loan' is meaningful."
)

#### 4. Removing URLs & Removing words and digits contain digits.

In [ ]:
# Remove URLs & Remove words and digits contain digits
note_nlp_step(
    "Removing URLs and digit-containing words",
    "the dataset has no narrative text or URLs used as model features."
)

#### 5. Removing Stopwords & Removing White spaces

In [ ]:
# Remove Stopwords
note_nlp_step(
    "Removing stopwords",
    "removing words could change the meaning of loan product labels."
)

In [ ]:
# Remove White spaces
text_preprocessing_status = {

    "Remove white spaces": "Not applicable: removing spaces could alter product labels.",
}
for step, explanation in text_preprocessing_status.items():
    print(f"{step}: {explanation}")

#### 6. Rephrase Text

In [ ]:
# Rephrase Text
text_preprocessing_status = {

    "Rephrase text": "Not applicable: no free-form text to rephrase.",
}
for step, explanation in text_preprocessing_status.items():
    print(f"{step}: {explanation}")

#### 7. Tokenization

In [ ]:
# Tokenization
text_preprocessing_status = {

    "Tokenization": "Not applicable: loan products are encoded as product indicators.",
}
for step, explanation in text_preprocessing_status.items():
    print(f"{step}: {explanation}")

#### 8. Text Normalization

In [ ]:
# Normalizing Text (i.e., Stemming, Lemmatization etc.)
# Trim leading and trailing whitespace from structured category labels.
# Keep internal spaces and punctuation in loan-product names.
for column in [
    "Type_of_Loan",
    "Credit_Mix",
    "Payment_of_Min_Amount",
    "Payment_Behaviour",
]:
    if column in df.columns:
        df[column] = df[column].astype("string").str.strip()

print("Outer whitespace trimmed from available category columns.")

##### Which text normalization technique have you used and why?

Answer Here.
I used trimming to remove accidental spaces around category values. I did not use stemming or lemmatization because the dataset has no free-form text; loan descriptions are structured product labels.

#### 9. Part of speech tagging

In [ ]:
# POS Taging
# POS tagging is for sentences, which this dataset does not contain.
print("POS tagging skipped: no free-form text columns are used.")

#### 10. Text Vectorization

In [ ]:
# Vectorizing Text
# Configure an encoder for categorical fields.
# It is fit within the model pipeline after the data split, not on the full dataset.
try:
    categorical_encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
    )
except TypeError:
    # Compatibility with older scikit-learn versions.
    categorical_encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse=False,
    )

print("One-hot encoder configured for use inside the training pipeline.")

##### Which text vectorization technique have you used and why?

Answer Here.
NLP vectorization such as TF-IDF is not appropriate because there are no free-text documents. The model pipeline uses one-hot encoding for categorical features. Loan products are represented as multi-hot product flags, so each product gets its own 0/1 feature. The encoder is fit during model training to avoid data leakage.

### 4. Feature Manipulation & Selection

#### 1. Feature Manipulation

In [ ]:
# Manipulate Features to minimize feature correlation and create new features
# Create a separate dataframe for model feature engineering.
df_model = df.copy()

# Define the known loan products and their new binary feature names.
loan_products = {
    "Auto Loan": "Loan_Auto",
    "Credit-Builder Loan": "Loan_Credit_Builder",
    "Debt Consolidation Loan": "Loan_Debt_Consolidation",
    "Home Equity Loan": "Loan_Home_Equity",
    "Mortgage Loan": "Loan_Mortgage",
    "Payday Loan": "Loan_Payday",
    "Personal Loan": "Loan_Personal",
    "Student Loan": "Loan_Student",
}

# Search the structured loan descriptions case-insensitively.
loan_text = df_model["Type_of_Loan"].astype("string").fillna("")
for product, feature_name in loan_products.items():
    df_model[feature_name] = loan_text.str.contains(
        product, case=False, regex=False, na=False
    ).astype("int8")

# Count how many distinct known loan-product types each customer has.
loan_flag_columns = list(loan_products.values())
df_model["Distinct_Loan_Product_Types"] = df_model[loan_flag_columns].sum(axis=1)

print("Created loan product indicators and Distinct_Loan_Product_Types.")

#### 2. Feature Selection

In [ ]:
# Select your features wisely to avoid overfitting
# Exclude identifiers, personal information, raw high-cardinality text, and the target.
excluded_columns = [
    "ID", "Customer_ID", "Month", "Name", "SSN",
    "Occupation", "Type_of_Loan", "Credit_Score",
]

# Keep Credit_Mix only if it is known to be independent of the target
# and available when predictions will be made.
INCLUDE_CREDIT_MIX = True
if not INCLUDE_CREDIT_MIX:
    excluded_columns.append("Credit_Mix")

# Separate predictors, target labels, and customer IDs for grouped splitting.
X = df_model.drop(columns=excluded_columns, errors="ignore").copy()
y = df_model["Credit_Score"].astype(str)
groups = df_model["Customer_ID"]

print(f"Rows: {len(X):,}")
print(f"Predictors before transformation: {X.shape[1]}")
print("Predictor columns:")
print(list(X.columns))

##### What all feature selection methods have you used  and why?

Answer Here.
I used domain-based screening to exclude IDs and personal information, and converted the high-cardinality loan-description column into product indicators. The correlation analysis found that Annual_Income and Monthly_Inhand_Salary are highly correlated (about 0.998), so I will remove the redundant salary field for the initial model and compare alternatives using group-aware cross-validation. After model training, permutation importance can help rank features by their effect on macro F1.

##### Which all features you found important and why?

Answer Here.
EDA suggests that payment delay, interest rate, credit inquiries, outstanding debt, credit-history age, income, and credit mix are associated with the target in this dataset. These are provisional signals from descriptive analysis, not confirmed model importance. The trained model’s permutation-importance results should be used to report final feature importance. Include Credit_Mix only if it is genuinely available before prediction and is not derived from the target.

### 5. Data Transformation

#### Do you think that your data needs to be transformed? If yes, which transformation have you used. Explain Why?

In [ ]:
# Transform Your data
# Create log-transformed versions of right-skewed financial amounts.
for column in ["Annual_Income", "Outstanding_Debt"]:
    if column not in X.columns:
        continue

    # Reject negative amounts instead of silently changing their meaning.
    if (X[column].dropna() < 0).any():
        raise ValueError(f"{column} contains negative values; investigate before transforming.")

    X[f"Log_{column}"] = np.log1p(X[column].clip(lower=0))

# Remove raw amounts after creating their transformed versions.
# Monthly salary is omitted because it is almost perfectly correlated with annual income.
X = X.drop(
    columns=["Annual_Income", "Outstanding_Debt", "Monthly_Inhand_Salary"],
    errors="ignore",
)

print("Log transformations created for income and outstanding debt.")
print(f"Predictors after transformation: {X.shape[1]}")

### 6. Data Scaling

In [ ]:
# Scaling your data
# Configure robust scaling for numeric predictors.
# Fit this scaler inside the model pipeline after the customer-aware data split.
numeric_scaler = RobustScaler()

print("RobustScaler configured; it will be fit on training data inside the pipeline.")

##### Which method have you used to scale you data and why?

Answer Here.
I selected RobustScaler, which scales using the median and interquartile range. This is less sensitive to extreme values than standard scaling. It should be fit within the training pipeline so validation and test records do not influence its statistics. Tree-based models generally need less scaling than Logistic Regression, but the shared preprocessing pipeline keeps the comparison consistent.

### 7. Dimesionality Reduction

##### Do you think that dimensionality reduction is needed? Explain Why?

Answer Here.


In [ ]:
# DImensionality Reduction (If needed)


##### Which dimensionality reduction technique have you used and why? (If dimensionality reduction done on dataset.)

Answer Here.

### 8. Data Splitting

In [ ]:
# Split your data to train and test. Choose Splitting ratio wisely.
# Data Splitting
# Keep all records belonging to a customer in either train or test, never both.
# Stratification aims to keep the target-class proportions similar.

import numpy as np
from sklearn.model_selection import StratifiedGroupKFold

# Confirm that the feature rows, labels, and customer groups are aligned.
if not (len(X) == len(y) == len(groups)):
    raise ValueError("X, y, and groups must contain the same number of rows.")

try:
    # Five folds give an approximately 80% train / 20% test split.
    splitter = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )
    train_idx, test_idx = next(splitter.split(X, y, groups=groups))

except (ImportError, AttributeError):
    # Fallback for older scikit-learn versions.
    from sklearn.model_selection import GroupShuffleSplit

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.20,
        random_state=42
    )
    train_idx, test_idx = next(splitter.split(X, y, groups=groups))

# Use positional indexing so the indices returned by the splitter are applied correctly.
X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()
groups_train = groups.iloc[train_idx].copy()
groups_test = groups.iloc[test_idx].copy()

# Check that customer groups do not overlap across the split.
overlap = set(groups_train.unique()).intersection(set(groups_test.unique()))
if overlap:
    raise RuntimeError("Customer leakage detected: some customers appear in both sets.")

# Display the split sizes and target distributions for verification.
print(f"Training rows: {len(X_train):,} ({len(X_train) / len(X):.1%})")
print(f"Testing rows:  {len(X_test):,} ({len(X_test) / len(X):.1%})")
print(f"Customers in training: {groups_train.nunique():,}")
print(f"Customers in testing:  {groups_test.nunique():,}")
print(f"Overlapping customers: {len(overlap)}")

print("\nTraining target proportions:")
print(y_train.value_counts(normalize=True).round(3))

print("\nTesting target proportions:")
print(y_test.value_counts(normalize=True).round(3))

##### What data splitting ratio have you used and why?

Answer Here.
The test set estimates performance on customers the model did not see during training. The printed target proportions let you check whether the split retained a reasonably similar class mix.

### 9. Handling Imbalanced Dataset

##### Do you think the dataset is imbalanced? Explain Why.

Answer Here.

In [ ]:
# Handling Imbalanced Dataset (If needed)

##### What technique did you use to handle the imbalance dataset and why? (If needed to be balanced)

Answer Here.

## ***7. ML Model Implementation***

### ML Model - 1

In [ ]:
# ML Model - 1 Implementation

# Fit the Algorithm

# Predict on the model
# Model 1: Logistic Regression baseline
# Preprocessing is kept inside the pipeline to prevent information leaking
# from the test data into training.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

# Identify numeric and categorical columns from the training features.
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

# Impute missing numeric values and scale numeric features.
numeric_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler()),
])

# Impute missing categories and encode categories as model-ready columns.
categorical_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

# Apply the appropriate preprocessing to each column type.
preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_preprocessor, numeric_features),
    ("categorical", categorical_preprocessor, categorical_features),
])

# class_weight="balanced" gives additional training weight to less common classes.
baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42,
    )),
])

# Train on the training split, then predict the untouched test split.
baseline_model.fit(X_train, y_train)
baseline_predictions = baseline_model.predict(X_test)

# Calculate multiple metrics so performance on smaller classes is visible.
def calculate_metrics(y_true, y_pred):
    """Return useful multiclass classification metrics."""
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced accuracy": balanced_accuracy_score(y_true, y_pred),
        "Macro F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "Weighted F1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }

baseline_scores = calculate_metrics(y_test, baseline_predictions)

print("Baseline test-set scores:")
display(pd.Series(baseline_scores, name="Score").round(3))

print("\nClass-by-class results:")
print(classification_report(y_test, baseline_predictions, zero_division=0))

# Plot a score chart for the baseline model.
plt.figure(figsize=(8, 4))
plt.bar(baseline_scores.keys(), baseline_scores.values(), color="steelblue")
plt.ylim(0, 1)
plt.ylabel("Score")
plt.title("Logistic Regression — Baseline Test Metrics")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart
# Visualize the baseline Logistic Regression evaluation metrics.
# baseline_scores is created in the model evaluation cell above.

import matplotlib.pyplot as plt
import pandas as pd

if "baseline_scores" not in globals() or not baseline_scores:
    raise ValueError(
        "Run the Logistic Regression model evaluation cell first "
        "to create baseline_scores."
    )

scores = pd.Series(baseline_scores, dtype="float64")

fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.bar(
    scores.index,
    scores.values,
    color=["steelblue", "darkorange", "seagreen", "mediumpurple"],
)

# Add each score above its bar for easy reading.
for bar, score in zip(bars, scores.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        score + 0.015,
        f"{score:.3f}",
        ha="center",
        va="bottom",
        fontsize=10,
    )

ax.set_ylim(0, 1.12)
ax.set_ylabel("Test-set score")
ax.set_title("Logistic Regression — Baseline Evaluation Metrics")
ax.tick_params(axis="x", rotation=15)
ax.grid(axis="y", linestyle="--", alpha=0.35)

plt.tight_layout()
plt.show()


#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 1 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)

# Fit the Algorithm

# Predict on the model
# Tune Logistic Regression using only the training data.
# Group-aware folds keep each customer's records together.

from sklearn.model_selection import RandomizedSearchCV, StratifiedGroupKFold

# Three folds provide a practical balance between validation reliability and runtime.
group_cv = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=42,
)

# Search a small, explicit set of Logistic Regression settings.
# C controls regularization: smaller values apply stronger regularization.
parameter_options = {
    "classifier__C": [0.01, 0.1, 1.0, 10.0],
    "classifier__class_weight": [None, "balanced"],
}

tuning_search = RandomizedSearchCV(
    estimator=baseline_model,
    param_distributions=parameter_options,
    n_iter=8,
    scoring="f1_macro",
    cv=group_cv,
    random_state=42,
    n_jobs=-1,
    refit=True,
    return_train_score=True,
    error_score="raise",
)

# Pass customer IDs to the cross-validation splitter.
tuning_search.fit(
    X_train,
    y_train,
    groups=groups_train.to_numpy(),
)

# The search refits the best settings on the full training split.
tuned_model = tuning_search.best_estimator_
tuned_predictions = tuned_model.predict(X_test)
tuned_scores = calculate_metrics(y_test, tuned_predictions)

print("Best cross-validation Macro F1:", round(tuning_search.best_score_, 3))
print("Best hyperparameters:", tuning_search.best_params_)

print("\nTuned test-set scores:")
display(pd.Series(tuned_scores, name="Score").round(3))

print("\nTuned class-by-class results:")
print(classification_report(y_test, tuned_predictions, zero_division=0))

##### Which hyperparameter optimization technique have you used and why?

Answer Here.
I used RandomizedSearchCV with three-fold StratifiedGroupKFold cross-validation. The search tested different values of Logistic Regression’s regularization parameter C and class_weight. I selected Macro F1 as the tuning score because it gives each credit-score class equal importance. Grouped folds keep each customer’s records together, reducing customer-level data leakage during validation.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Answer Here.
Tuning does not guarantee better test results. I compare the tuned model with the baseline on the same untouched test set. Use the printed Macro F1 scores and chart below to report whether performance improved; the result should come from your notebook run.

### ML Model - 2

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 1 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)

# Fit the Algorithm

# Predict on the model

##### Which hyperparameter optimization technique have you used and why?

Answer Here.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Answer Here.

#### 3. Explain each evaluation metric's indication towards business and the business impact pf the ML model used.

Answer Here.

### ML Model - 3

In [ ]:
# ML Model - 3 Implementation

# Fit the Algorithm

# Predict on the model

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 3 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)

# Fit the Algorithm

# Predict on the model

##### Which hyperparameter optimization technique have you used and why?

Answer Here.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Answer Here.

### 1. Which Evaluation metrics did you consider for a positive business impact and why?

Answer Here.
I considered Macro F1 the main comparison metric because the target classes are imbalanced and Macro F1 gives each class equal weight. I also reviewed balanced accuracy, class-wise precision and recall, and the confusion matrix.
In a credit-score setting, confusing a Poor score with Good may lead to a riskier recommendation, while confusing Good with Poor may miss a suitable customer opportunity. Those errors have different business costs, so overall accuracy alone is not enough. The model should support human review and should not be treated as a complete credit decision by itself.

### 2. Which ML model did you choose from the above created models as your final prediction model and why?

Answer Here.
Logistic Regression is the only model trained so far, use the tuned Logistic Regression pipeline as the current final model. Its preprocessing is included in the pipeline, it handles numeric and categorical features, and it is comparatively interpretable. Confirm its cross-validation Macro F1 and held-out test results before describing it as the best-performing model.

### 3. Explain the model which you have used and the feature importance using any model explainability tool?

Answer Here.
Logistic Regression estimates how each feature relates to the likelihood of each credit-score class. The preprocessing pipeline imputes missing values, scales numerical variables, and one-hot encodes categorical variables before fitting the classifier.
I use permutation importance to explain the model. It measures how much Macro F1 changes when one feature is shuffled. A larger positive value means the model relied more on that feature for its predictions. This shows model reliance, not cause and effect; correlated features can share or obscure importance.

## ***8.*** ***Future Work (Optional)***

### 1. Save the best performing ml model in a pickle file or joblib file format for deployment process.


In [ ]:
from pathlib import Path
import joblib

# The notebook is in Downloads, while the model is in the credit_score_api folder.
deployment_dir = Path.home() / "Downloads" / "credit_score_api"
bundle_path = deployment_dir / "model_bundle.joblib"

if not bundle_path.exists():
    raise FileNotFoundError(
        f"Model bundle not found at: {bundle_path}\n"
        f"Files in that folder: {list(deployment_dir.glob('*'))}"
    )

bundle = joblib.load(bundle_path)

print("Bundle keys:", list(bundle.keys()))
print("Number of expected features:", len(bundle["feature_columns"]))

In [ ]:
# Select a fitted model and save it with the feature schema required by the API.

from pathlib import Path
import joblib
from sklearn.utils.validation import check_is_fitted

# Prefer the best model found by hyperparameter tuning.
if "tuning_search" in globals() and hasattr(tuning_search, "best_estimator_"):
    final_model = tuning_search.best_estimator_
elif "tuned_model" in globals():
    final_model = tuned_model
elif "baseline_model" in globals():
    final_model = baseline_model
else:
    raise RuntimeError(
        "No trained model was found. Run the model training and tuning cells first."
    )

# Confirm that the selected estimator has actually been fitted.
check_is_fitted(final_model)

# The API needs the exact input columns and order used during training.
if "X_train" not in globals():
    raise RuntimeError("X_train is missing. Run the data-splitting cell first.")

# Save the bundle where the deployment instructions expect to find it.
deployment_dir = Path("credit_score_api")
deployment_dir.mkdir(parents=True, exist_ok=True)

model_bundle = {
    "model": final_model,
    "feature_columns": X_train.columns.tolist(),
}

model_path = deployment_dir / "model_bundle.joblib"
joblib.dump(model_bundle, model_path, compress=3)

print(f"Saved fitted model and feature schema to: {model_path.resolve()}")

### 2. Again Load the saved model file and try to predict unseen data for a sanity check.


In [ ]:
# Load the File and predict unseen data.
# Load the saved model bundle and make predictions on held-out feature rows.

from pathlib import Path
import joblib
import pandas as pd

model_path = Path("credit_score_api/model_bundle.joblib")

if not model_path.exists():
    raise FileNotFoundError(
        f"Model bundle not found at {model_path.resolve()}. "
        "Run the corrected model-save cell first."
    )

# Load both the fitted pipeline and its expected feature-column schema.
model_bundle = joblib.load(model_path)
loaded_model = model_bundle["model"]
required_features = model_bundle["feature_columns"]

if "X_test" not in globals():
    raise RuntimeError("X_test is missing. Run the data-splitting cell first.")

# Check that the held-out data contains every feature required by the model.
missing_features = sorted(set(required_features) - set(X_test.columns))
if missing_features:
    raise ValueError(f"X_test is missing required features: {missing_features}")

# Arrange columns in the exact order used when the model was trained.
X_unseen = X_test.loc[:, required_features].head(5).copy()

# Predict the score category for the sample rows.
predicted_classes = loaded_model.predict(X_unseen)

sanity_results = pd.DataFrame(
    {"Predicted Credit Score": predicted_classes},
    index=X_unseen.index,
)

# Include class probabilities when the model supports them.
if hasattr(loaded_model, "predict_proba"):
    probabilities = loaded_model.predict_proba(X_unseen)
    class_names = loaded_model.classes_

    for column_index, class_name in enumerate(class_names):
        sanity_results[f"Probability: {class_name}"] = probabilities[:, column_index]

display(sanity_results)

### ***Congrats! Your model is successfully created and ready for deployment on a live server for a real user interaction !!!***

# **Conclusion**

Write the conclusion here.

### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***